# Exercise 2 — DML and TMLE step by step

**Estimated time:** 15 minutes  
**Lecture notation:** $Q(w,a)=\mathbb E[Y\mid W=w,A=a]$, $\pi(a\mid w)=\mathbb P(A=a\mid W=w)$, $\psi=\mathbb E[Y^1-Y^0]$.

## Learning goals

1. Implement DML as cross-fitted AIPW / one-step estimation.
2. Implement TMLE as a targeted plug-in estimator.
3. Compare influence-function-based standard errors.

We use a binary outcome so that TMLE can visibly respect the natural bounds of the parameter.

## Colab instructions

Use **File → Save a copy in Drive** before editing. You can also download the notebook with **File → Download → Download .ipynb**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit, logit
from scipy.optimize import minimize_scalar

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(321)
plt.rcParams["figure.figsize"] = (8, 4.5)


## 1. Binary-outcome data

The estimand is the risk difference

$$
\psi = \mathbb E\{Q(W,1)-Q(W,0)\}=\mathbb E[Y^1-Y^0].
$$

The variable `M` is post-treatment and is included to create a conceptual trap.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def simulate_binary_ate(n=3000, seed=1, exercise=False):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=n)
    W2 = rng.normal(size=n)
    W3 = rng.normal(size=n)
    W4 = rng.normal(size=n) if exercise else np.zeros(n)

    lin_pi = -0.2 + 0.7 * W1 - 0.6 * W2 + 0.3 * W3 + (0.45 * W4 if exercise else 0)
    pi = np.clip(sigmoid(lin_pi), 0.04, 0.96)
    A = rng.binomial(1, pi)

    tau_part = 0.8 + 0.4 * W1 - 0.2 * W2 + (0.25 * W4 if exercise else 0)
    base = -1.0 + 0.8 * np.sin(W1) - 0.5 * W2 + 0.3 * W3
    p0 = sigmoid(base)
    p1 = sigmoid(base + tau_part)

    M = 0.8 * A + 0.5 * W1 + rng.normal(scale=0.7, size=n)
    p_obs = np.where(A == 1, p1, p0)
    Y = rng.binomial(1, p_obs)

    df = pd.DataFrame({"W1": W1, "W2": W2, "W3": W3, "W4": W4, "A": A, "M": M, "Y": Y, "pi_true": pi, "p0_true": p0, "p1_true": p1})
    df["tau_true"] = df["p1_true"] - df["p0_true"]
    return df


def true_ate_binary(df):
    return np.mean(df["p1_true"] - df["p0_true"])

df = simulate_binary_ate(n=2500, seed=7, exercise=False)
ref = simulate_binary_ate(n=100_000, seed=77, exercise=False)
psi_true = true_ate_binary(ref)
print(df.head())
print(f"Approximate true risk difference psi = {psi_true:.3f}")


## 2. Completed DML example

DML uses sample splitting / cross-fitting. For each observation, $\widehat Q$ and $\widehat\pi$ are trained on other folds, and the AIPW score is evaluated out of sample.


In [ ]:
def crossfit_nuisances(df, W_cols, n_splits=3, seed=123,
                      q_model=None, pi_model=None):
    if q_model is None:
        q_model = RandomForestClassifier(n_estimators=100, min_samples_leaf=30, random_state=seed)
    if pi_model is None:
        pi_model = RandomForestClassifier(n_estimators=100, min_samples_leaf=30, random_state=seed + 1)

    n = len(df)
    Q0_hat = np.zeros(n)
    Q1_hat = np.zeros(n)
    pi_hat = np.zeros(n)

    X = df[W_cols]
    XA = df[W_cols + ["A"]]
    Y = df["Y"].values
    A = df["A"].values

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, test_idx in kf.split(df):
        q = clone(q_model)
        p = clone(pi_model)

        q.fit(XA.iloc[train_idx], Y[train_idx])
        p.fit(X.iloc[train_idx], A[train_idx])

        X_test = X.iloc[test_idx].copy()
        X1 = X_test.copy(); X1["A"] = 1
        X0 = X_test.copy(); X0["A"] = 0
        Q1_hat[test_idx] = q.predict_proba(X1[W_cols + ["A"]])[:, 1]
        Q0_hat[test_idx] = q.predict_proba(X0[W_cols + ["A"]])[:, 1]
        pi_hat[test_idx] = p.predict_proba(X_test)[:, 1]

    return np.clip(Q0_hat, 1e-4, 1-1e-4), np.clip(Q1_hat, 1e-4, 1-1e-4), np.clip(pi_hat, 0.02, 0.98)


def dml_ate_from_nuisances(Y, A, Q0_hat, Q1_hat, pi_hat):
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)
    phi = (Q1_hat - Q0_hat) + (A / pi_hat - (1 - A) / (1 - pi_hat)) * (Y - Q_A_hat)
    psi = phi.mean()
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return psi, sd_phi, se, phi

W_cols = ["W1", "W2", "W3"]
Q0_hat, Q1_hat, pi_hat = crossfit_nuisances(df, W_cols=W_cols, n_splits=3)
psi_dml, sd_dml, se_dml, phi_dml = dml_ate_from_nuisances(df["Y"].values, df["A"].values, Q0_hat, Q1_hat, pi_hat)

print(f"DML estimate = {psi_dml:.3f}")
print(f"IF SD = {sd_dml:.3f}")
print(f"SE = {se_dml:.3f}")
print(f"95% CI = [{psi_dml - 1.96*se_dml:.3f}, {psi_dml + 1.96*se_dml:.3f}]")


## 3. Completed TMLE example

TMLE starts with initial $\widehat Q^0$ and $\widehat\pi$, then targets $\widehat Q^0$ by fitting a one-dimensional fluctuation. For the ATE risk difference, the clever covariate is

$$
H(A,W)=\frac{A}{\widehat\pi(1\mid W)}-\frac{1-A}{\widehat\pi(0\mid W)}.
$$

The final estimator is the targeted plug-in:

$$
\widehat\psi_{\rm TMLE}=\frac{1}{n}\sum_i\{\widehat Q^\star(W_i,1)-\widehat Q^\star(W_i,0)\}.
$$


In [ ]:
def tmle_update_binary(Y, A, Q0_hat, Q1_hat, pi_hat):
    eps = 1e-6
    Q0_hat = np.clip(Q0_hat, eps, 1 - eps)
    Q1_hat = np.clip(Q1_hat, eps, 1 - eps)
    pi_hat = np.clip(pi_hat, 0.02, 0.98)

    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)
    H_obs = A / pi_hat - (1 - A) / (1 - pi_hat)
    offset = logit(Q_A_hat)

    def neg_loglik(epsilon):
        q_star_obs = expit(offset + epsilon * H_obs)
        return -np.sum(Y * np.log(q_star_obs + eps) + (1 - Y) * np.log(1 - q_star_obs + eps))

    res = minimize_scalar(neg_loglik, bracket=(-1, 1), method="brent")
    epsilon_hat = res.x

    H1 = 1 / pi_hat
    H0 = -1 / (1 - pi_hat)
    Q1_star = expit(logit(Q1_hat) + epsilon_hat * H1)
    Q0_star = expit(logit(Q0_hat) + epsilon_hat * H0)
    Q_A_star = np.where(A == 1, Q1_star, Q0_star)

    psi = np.mean(Q1_star - Q0_star)
    phi = H_obs * (Y - Q_A_star) + (Q1_star - Q0_star) - psi
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return psi, sd_phi, se, epsilon_hat, Q0_star, Q1_star, phi

psi_tmle, sd_tmle, se_tmle, eps_hat, Q0_star, Q1_star, phi_tmle = tmle_update_binary(df["Y"].values, df["A"].values, Q0_hat, Q1_hat, pi_hat)

print(f"TMLE estimate = {psi_tmle:.3f}")
print(f"Targeting epsilon = {eps_hat:.4f}")
print(f"IF SD = {sd_tmle:.3f}")
print(f"SE = {se_tmle:.3f}")
print(f"95% CI = [{psi_tmle - 1.96*se_tmle:.3f}, {psi_tmle + 1.96*se_tmle:.3f}]")
print(f"Mean IF after targeting = {phi_tmle.mean():.3e}")


In [ ]:
example_results = pd.DataFrame({
    "estimator": ["true psi", "DML", "TMLE"],
    "estimate": [psi_true, psi_dml, psi_tmle],
    "se": [np.nan, se_dml, se_tmle],
})
example_results


In [ ]:
fig, ax = plt.subplots()
ypos = np.arange(len(example_results))
ax.scatter(example_results["estimate"], ypos)
for j, row in example_results.iterrows():
    if np.isfinite(row["se"]):
        ax.errorbar(row["estimate"], j, xerr=1.96 * row["se"], fmt="none", capsize=4)
ax.axvline(psi_true, linestyle="--", label="true psi")
ax.set_yticks(ypos)
ax.set_yticklabels(example_results["estimator"])
ax.set_xlabel("Risk difference estimate")
ax.set_title("DML and TMLE are asymptotically equivalent but implemented differently")
ax.legend()
plt.show()


# Student task: complete DML and TMLE on new data

The new data adds a fourth baseline confounder `W4` and still includes a post-treatment variable `M`. The goal is the **total risk difference** $\mathbb E[Y^1-Y^0]$.


In [ ]:
df_ex = simulate_binary_ate(n=3000, seed=2026, exercise=True)
ref_ex = simulate_binary_ate(n=100_000, seed=2027, exercise=True)
psi_true_ex = true_ate_binary(ref_ex)

print(df_ex.head())
print(f"Approximate true total risk difference = {psi_true_ex:.3f}")


## Task 1 — Choose $W$

The code will run for many choices. The causal question determines the right choice.


In [ ]:
# TODO: choose covariates for Q and pi.
# Candidate variables: "W1", "W2", "W3", "W4", "M".
# Hint: M is post-treatment.
W_cols_ex = ["W1", "W2", "W3", "W4"]  # <-- edit this


## Task 2 — Cross-fit nuisance functions

This part is supplied. Your choices enter through `W_cols_ex`.


In [ ]:
Q0_ex, Q1_ex, pi_ex = crossfit_nuisances(df_ex, W_cols=W_cols_ex, n_splits=3, seed=2026)
print("Q0 range:", Q0_ex.min(), Q0_ex.max())
print("Q1 range:", Q1_ex.min(), Q1_ex.max())
print("pi range:", pi_ex.min(), pi_ex.max())


## Task 3 — Complete the DML score

Complete the two conceptual lines. Wrong choices can still run, but they answer a different question or fail to debias.


In [ ]:
def dml_score_student(Y, A, Q0_hat, Q1_hat, pi_hat):
    pi_hat = np.clip(pi_hat, 0.02, 0.98)

    # TODO 1: choose Q(W_i,A_i), the prediction under the observed treatment.
    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)  # <-- edit only if you want to test alternatives

    # TODO 2: write the AIPW / DML score.
    phi = (Q1_hat - Q0_hat) + (A / pi_hat - (1 - A) / (1 - pi_hat)) * (Y - Q_A_hat)  # <-- edit if needed

    psi = phi.mean()
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return psi, sd_phi, se, phi

psi_dml_ex, sd_dml_ex, se_dml_ex, phi_dml_ex = dml_score_student(
    df_ex["Y"].values, df_ex["A"].values, Q0_ex, Q1_ex, pi_ex
)
print(f"Your DML estimate = {psi_dml_ex:.3f}")
print(f"IF SD = {sd_dml_ex:.3f}; SE = {se_dml_ex:.3f}")


## Task 4 — Complete the TMLE targeting step

The key conceptual objects are:

- observed clever covariate $H(A,W)$ for fitting the fluctuation;
- counterfactual clever covariates $H(1,W)$ and $H(0,W)$ for updating $Q(W,1)$ and $Q(W,0)$.


In [ ]:
def tmle_student(Y, A, Q0_hat, Q1_hat, pi_hat):
    eps = 1e-6
    Q0_hat = np.clip(Q0_hat, eps, 1 - eps)
    Q1_hat = np.clip(Q1_hat, eps, 1 - eps)
    pi_hat = np.clip(pi_hat, 0.02, 0.98)

    Q_A_hat = np.where(A == 1, Q1_hat, Q0_hat)
    offset = logit(Q_A_hat)

    # TODO 1: clever covariate observed in the data.
    H_obs = A / pi_hat - (1 - A) / (1 - pi_hat)  # <-- edit if needed

    def neg_loglik(epsilon):
        q_star_obs = expit(offset + epsilon * H_obs)
        return -np.sum(Y * np.log(q_star_obs + eps) + (1 - Y) * np.log(1 - q_star_obs + eps))

    epsilon_hat = minimize_scalar(neg_loglik, bracket=(-1, 1), method="brent").x

    # TODO 2: clever covariates for the two counterfactual predictions.
    H1 = 1 / pi_hat            # H(A=1,W)
    H0 = -1 / (1 - pi_hat)     # H(A=0,W)

    Q1_star = expit(logit(Q1_hat) + epsilon_hat * H1)
    Q0_star = expit(logit(Q0_hat) + epsilon_hat * H0)
    Q_A_star = np.where(A == 1, Q1_star, Q0_star)

    psi = np.mean(Q1_star - Q0_star)
    phi = H_obs * (Y - Q_A_star) + (Q1_star - Q0_star) - psi
    sd_phi = phi.std(ddof=1)
    se = sd_phi / np.sqrt(len(Y))
    return psi, sd_phi, se, epsilon_hat, phi

psi_tmle_ex, sd_tmle_ex, se_tmle_ex, eps_ex, phi_tmle_ex = tmle_student(
    df_ex["Y"].values, df_ex["A"].values, Q0_ex, Q1_ex, pi_ex
)
print(f"Your TMLE estimate = {psi_tmle_ex:.3f}")
print(f"Targeting epsilon = {eps_ex:.4f}")
print(f"IF SD = {sd_tmle_ex:.3f}; SE = {se_tmle_ex:.3f}")
print(f"Mean IF after targeting = {phi_tmle_ex.mean():.3e}")


In [ ]:
student_results = pd.DataFrame({
    "estimator": ["true total risk difference", "DML", "TMLE"],
    "estimate": [psi_true_ex, psi_dml_ex, psi_tmle_ex],
    "se": [np.nan, se_dml_ex, se_tmle_ex],
})
student_results


In [ ]:
fig, ax = plt.subplots()
ypos = np.arange(len(student_results))
ax.scatter(student_results["estimate"], ypos)
for j, row in student_results.iterrows():
    if np.isfinite(row["se"]):
        ax.errorbar(row["estimate"], j, xerr=1.96 * row["se"], fmt="none", capsize=4)
ax.axvline(psi_true_ex, linestyle="--", label="true total effect")
ax.set_yticks(ypos)
ax.set_yticklabels(student_results["estimator"])
ax.set_xlabel("Risk difference estimate")
ax.set_title("Your DML and TMLE implementations")
ax.legend()
plt.show()


## Questions for discussion

1. Why does DML evaluate nuisance functions out of sample?
2. Why does TMLE update $\widehat Q$ rather than only adding a correction to $\widehat\psi$?
3. Why should the empirical mean of the final TMLE influence-function score be close to zero?
4. What happens if you add `M` to `W_cols_ex`?
